# Consolidação dos acidentes

Este notebook organiza e limpa as informações públicas de acidentes nas rodovias estaduais paulistas, armazenando-as em ```Processed/Acidentes.csv```.

In [36]:
from pathlib import Path
import pandas as pd
import re
import os

pasta_in = Path("In")
pasta_out = Path("Processed")

pasta_out.mkdir(parents=True, exist_ok=True)

arquivos_entrada = []
arquivo_saida = pasta_out / 'Acidentes.csv'

for dirpath, _, filename in os.walk(pasta_in):
    for file in filename:
        if "acidentes_pistas" in file:
            arquivos_entrada.append(pasta_in / file)

In [37]:
from pandas import DataFrame

acidentes = DataFrame()
for arquivo_entrada in arquivos_entrada:
    acidentes = pd.concat([acidentes, pd.read_csv(arquivo_entrada, encoding="utf-8-sig", dtype=str, keep_default_na=False)], ignore_index=True)

In [38]:
def converter_veiculos_coluna(cell: str):
    try:
        pairs = {}

        vehicles = cell.split("|")
        for vech in vehicles:
            partial = vech.split("=")
            pairs[partial[0]] = int(partial[1])
        
        return pairs
    except:
        return {}

def normalizar_classe(s):
    if pd.isna(s):
        return "NÃO INFORMADO"
    
    s = s.upper().strip()
    
    if re.search(r"OBJETO LAN.ADO", s):
        return "OBJETO LANÇADO CONTRA O VEÍCULO"
    
    return s

def normalizar_subclasse(s):
    if pd.isna(s):
        return "NÃO INFORMADO"
    
    s = s.upper().strip()
    
    regras = [
        (r"DEFENSA|BARREIRA|SUBMARINO", "CHOQUE-DEFENSA/BARREIRA"),
        (r"DRENAGEM", "CHOQUE-ELEMENTO DE DRENAGEM"),
        (r"TALUDE|BARRANCO|CORTE", "CHOQUE-TALUDE/BARRANCO"),
        (r"MEIO.?FIO|CALÇAMENTO", "CHOQUE-MEIO FIO"),
        (r"ÁRVORE|ARVORE", "CHOQUE-ÁRVORE"),
        (r"POSTE", "CHOQUE-POSTE"),
        (r"BURACO", "CHOQUE-BURACO"),
        (r"OBJETO.*PISTA|OBJETO SOBRE A VIA|VEÍCULO PARADO NA PISTA", "CHOQUE-OBJETO NA PISTA"),
        (r"VEÍCULO PARADO NO ACOSTAMENTO", "CHOQUE-VEÍCULO PARADO NO ACOSTAMENTO"),
        (r"OAE|PILAR|VIADUTO|PONTE", "CHOQUE-OAE (PONTE/VIADUTO)"),
        (r"PRAÇA|CABINE|CANCELA|PEDÁGIO", "CHOQUE-PRAÇA DE PEDÁGIO"),
        (r"SINALIZAÇÃO|EQUIPAMENTO|PAINEL", "CHOQUE-SINALIZAÇÃO/EQUIPAMENTO"),
        (r"CERCA|ALAMBRADO|MOURÃO", "CHOQUE-CERCAS/ALAMBRADOS"),
        (r"EDIFICAÇÃO|ILHA|MATACÃO|OUTROS|NÃO IDENTIF", "CHOQUE-OUTROS"),
        
        (r"^FRONTAL$|COLIS.O-FRONTAL", "COLISÃO-FRONTAL"),
        (r"^TRASEIRA$|COLIS.O-TRASEIRA", "COLISÃO-TRASEIRA"),
        (r"^LATERAL$|COLIS.O-LATERAL", "COLISÃO-LATERAL"),
        (r"^TRANSVERSAL$|COLIS.O-TRANSVERSAL", "COLISÃO-TRANSVERSAL"),
        
        (r"^TOMBAMENTO$", "TOMBAMENTO"),
        (r"TOMBAMENTO-MOTO", "TOMBAMENTO-MOTO"),
        (r"TOMBAMENTO.*PESAD", "TOMBAMENTO-VEÍCULO PESADO"),
        (r"TOMBAMENTO-BICICLETA", "TOMBAMENTO-BICICLETA"),
        
        (r"^CAPOTAMENTO$", "CAPOTAMENTO"),
        (r"^ENGAVETAMENTO$", "ENGAVETAMENTO"),
        
        (r"SUICID|SUICÍD", "ATROP. PEDESTRE-SUICÍDIO"),
        (r"CICLISTA", "ATROP. PEDESTRE-CICLISTA"),
        (r"ATROP\.? PEDESTRE|DE PEDESTRE|PEDESTRE USUÁRIO", "ATROP. PEDESTRE-OUTROS"),
        
        (r"ANIMAL.*SILVESTRE.*GRANDE", "ATROP. ANIMAL-SILVESTRE GRANDE"),
        (r"ANIMAL.*SILVESTRE.*M.DIO", "ATROP. ANIMAL-SILVESTRE MÉDIO"),
        (r"ANIMAL.*SILVESTRE.*PEQUENO", "ATROP. ANIMAL-SILVESTRE PEQUENO"),
        (r"ANIMAL.*DOM.STICO.*GRANDE", "ATROP. ANIMAL-DOMÉSTICO GRANDE"),
        (r"ANIMAL.*DOM.STICO.*M.DIO", "ATROP. ANIMAL-DOMÉSTICO MÉDIO"),
        (r"ANIMAL.*DOM.STICO.*PEQUENO", "ATROP. ANIMAL-DOMÉSTICO PEQUENO"),
        (r"ANIMAL", "ATROP. ANIMAL-OUTROS"),
        
        (r"^QUEDA-MOTO$", "QUEDA-MOTO"),
        (r"^QUEDA-CICLISTA$", "QUEDA-CICLISTA"),
        (r"RIBANCEIRA|EM RIBANCEIRA", "QUEDA-RIBANCEIRA/OAE"),
        (r"QUEDA-CARGA", "QUEDA-CARGA"),
        (r"^QUEDA$|TABLUDE", "QUEDA-OUTROS"),
        
        (r"LAN.ADO", "OBJETO LANÇADO CONTRA O VEÍCULO"),
        (r"INC.NDIO", "INCÊNDIO"),
        (r"SA.DA DE PISTA", "SAÍDA DE PISTA"),
    ]
    
    for padrao, categoria in regras:
        if re.search(padrao, s):
            return categoria
    
    return "OUTROS/NÃO CLASSIFICADO"

In [39]:
df_dicts = acidentes.VEICULOS_ENVOLVIDOS.apply(converter_veiculos_coluna)
df_expanded = pd.json_normalize(df_dicts).fillna(0)

acidentes = pd.concat([acidentes, df_expanded], axis=1)
acidentes.drop(columns="VEICULOS_ENVOLVIDOS")

acidentes["CLASSE"] = acidentes["CLASSE"].apply(normalizar_classe)
acidentes["SUBCLASSE"] = acidentes["SUBCLASSE"].apply(normalizar_subclasse)

acidentes["NUMVEICULOS"] = acidentes.loc[:, 'AUTOMÓVEL' : "CARRETINHA"].sum(axis=1)
acidentes = acidentes.loc[acidentes.RODOVIA.str.contains("330")]
acidentes = acidentes.drop(columns=["_id", "VITIMAS_SEM_INFO", "VITIMA_ILESA","VISIBILIDADE","CONDICAO_METERIOLOGICA","VEICULOS_ENVOLVIDOS"])
acidentes = acidentes.sort_values([acidentes.DATA.name, acidentes.HORA.name], ascending=True)

acidentes.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')